In [0]:
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("bronze_adls", "")
dbutils.widgets.text("silver_adls", "")
dbutils.widgets.text("gold_adls", "")

In [0]:
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
bronze_adls = dbutils.widgets.get("bronze_adls")
silver_adls = dbutils.widgets.get("silver_adls")
gold_adls = dbutils.widgets.get("gold_adls")

In [0]:
# Retrieve the task value from the previous task (bronze)
bronze_output = dbutils.jobs.taskValues.get(taskKey="Bronze", key="bronze_output")

# Access individual variables
start_date = bronze_output.get("start_date", "")
bronze_adls = bronze_output.get("bronze_adls", "")
silver_adls = bronze_output.get("silver_adls", "")

print(f"Start Date: {start_date}, Bronze ADLS: {bronze_adls}")

In [0]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

In [0]:
# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"{bronze_adls}/{start_date}_earthquake_data.json")

In [0]:
df.head()

Row(geometry=Row(coordinates=[-120.282165527344, 36.2236671447754, 10.9799995422363], type='Point'), id='nc75323272', properties=Row(alert=None, cdi=None, code='75323272', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc75323272&format=geojson', dmin=0.05594, felt=None, gap=116, ids=',nc75323272,', mag=2.36, magType='md', mmi=None, net='nc', nst=27, place='11 km NE of Coalinga, CA', rms=0.09, sig=86, sources=',nc,', status='automatic', time=1772755018120, title='M 2.4 - 11 km NE of Coalinga, CA', tsunami=0, type='earthquake', types=',focal-mechanism,nearby-cities,origin,phase-data,scitech-link,', tz=None, updated=1772763141045, url='https://earthquake.usgs.gov/earthquakes/eventpage/nc75323272'), type='Feature')

In [0]:
# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)

In [0]:
df.head()

Row(id='nc75323272', longitude=-120.282165527344, latitude=36.2236671447754, elevation=10.9799995422363, title='M 2.4 - 11 km NE of Coalinga, CA', place_description='11 km NE of Coalinga, CA', sig=86, mag=2.36, magType='md', time=1772755018120, updated=1772763141045)

In [0]:
# Validate data: Check for missing or null values
df = (
    df
    .withColumn('longitude', when(isnull(col('longitude')), 0).otherwise(col('longitude')))
    .withColumn('latitude', when(isnull(col('latitude')), 0).otherwise(col('latitude')))
    .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))
)

In [0]:
# Convert 'time' and 'updated' to timestamp from Unix time
df = (
    df
    .withColumn('time', (col('time') / 1000).cast(TimestampType()))
    .withColumn('updated', (col('updated') / 1000).cast(TimestampType()))
)

In [0]:
df.head()

Row(id='nc75323272', longitude=-120.282165527344, latitude=36.2236671447754, elevation=10.9799995422363, title='M 2.4 - 11 km NE of Coalinga, CA', place_description='11 km NE of Coalinga, CA', sig=86, mag=2.36, magType='md', time=datetime.datetime(2026, 3, 5, 23, 56, 58, 120000), updated=datetime.datetime(2026, 3, 6, 2, 12, 21, 45000))

In [0]:
# Save the transformed DataFrame to the Silver container
silver_output_path = f"{silver_adls}/earthquake_events_silver/"

In [0]:
# Append DataFrame to Silver container in Parquet format
df.write.mode('append').parquet(silver_output_path)

In [0]:
dbutils.jobs.taskValues.set(key = "silver_output", value = silver_output_path)